# Мешок слов, n-граммы и TF-IDF

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Строить мешок слов, n-граммы и объяснять веса TF-IDF.

## Считаем, что встретилось

Мешок слов описывает документ счётчиками слов. Для словаря [кот,спит] фраза «кот кот спит» даёт [2,1]. Порядок потерян: «кот видит пса» и «пса видит кот» могут получить одинаковые счётчики.

N-грамма — последовательность n токенов. Биграммы фразы «кот не спит»: «кот не» и «не спит». Они сохраняют часть локального порядка, но увеличивают словарь и не заменяют полную модель структуры языка.

## Разобранный пример: распространённое и редкое

TF отражает частоту слова в документе. IDF уменьшает вес слов, встречающихся во многих документах. Одно распространённое соглашение: $idf(t)=\ln((1+N)/(1+df(t)))+1$, где N — число обучающих документов, df — число документов со словом. Это сглаженная формула TfidfVectorizer в примере.

В двух документах «кот спит», «кот играет» слово «кот» имеет df=2 и idf=1; «спит» имеет df=1 и idf=ln(3/2)+1>1. Затем часто нормируют весь вектор документа. Поэтому итоговый вес зависит не только от одного слова, но и от остальных слов документа.

## Обучение преобразователя

CountVectorizer строит словарь и счётчики, TfidfVectorizer дополнительно учит idf. Их fit выполняется только на train. transform использует известный словарь: полностью неизвестные слова не создают новые столбцы. ngram_range=(1,2) включает слова и биграммы.

Результат обычно разреженный: хранятся в основном ненулевые значения. Преобразовывать огромную таблицу в плотный массив ради печати опасно; toarray используем только для двух маленьких учебных документов.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

`norm=None` отключает дополнительную нормировку TF-IDF, чтобы можно было проверить исходные веса вручную. `get_feature_names_out()` возвращает слова в порядке столбцов, `idf_` — выученные IDF в том же порядке. Результат `transform` хранится как разреженная таблица: `nnz` считает ненулевые элементы, `toarray()` показывает обычную таблицу. У полностью неизвестного слова ненулевых элементов нет. Векторизаторы имеют своё правило выделения слов: по умолчанию односимвольные слова пропускаются; наши слова длиннее.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
texts = ["кот спит", "кот играет"]
counts = CountVectorizer().fit(texts)
assert counts.transform(["кот кот спит"]).sum() == 3
weights = TfidfVectorizer(norm=None).fit(texts)
names = weights.get_feature_names_out().tolist()
assert weights.idf_[names.index("спит")] > weights.idf_[names.index("кот")]
assert weights.transform(["неизвестное"]).nnz == 0
print(names, weights.transform(texts).toarray())


## Типичные ошибки

Не путайте df и частоту внутри текста. Не переводите большую разреженную таблицу в плотную без оценки памяти.

## Задачи

### 1. Базовая

Теряется ли порядок в мешке слов?

<details>
<summary>Подсказка</summary>

Сохраняются счётчики.

</details>



### 2. Базовая

Что означает df?

<details>
<summary>Подсказка</summary>

Счёт относится к документам.

</details>



### 3. Прикладная

Постройте счётчики «кот кот спит» для [кот,спит].

<details>
<summary>Подсказка</summary>

Повторы учитываются.

</details>



### 4. Прикладная

Назовите биграммы «модель не ошиблась».

<details>
<summary>Подсказка</summary>

Возьмите соседние пары.

</details>



### 5. Олимпиадная

Почему idf нельзя обучать отдельно на тесте?

<details>
<summary>Подсказка</summary>

Изменятся веса тех же столбцов.

</details>



### 6. Олимпиадная

Почему слово, повторённое сто раз в одном документе, не даёт df=100?

<details>
<summary>Подсказка</summary>

df считает присутствие по документам.

</details>



## Итоги и самопроверка

Вручную сравните idf общего и редкого слова. Что добавляют биграммы и какая цена у этого расширения?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Да, глобальный порядок слов не записан.

### Задача 2

Число обучающих документов, содержащих слово, не число всех повторов слова.

### Задача 3

[2,1].

### Задача 4

«модель не», «не ошиблась».

### Задача 5

Модель получит другую шкалу, а информация о составе проверки войдёт в подготовку.

### Задача 6

Оно добавляет единицу df для этого документа; повторы влияют на TF.